## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Employee attitudes and performance: two sets of measures

**Canonical Correlation Analysis** · a worked example from the StatNex teaching guide

**The story.** A company measured 600 employees twice over: three attitude measures (engagement, autonomy, share of training completed) and four performance scores (sales, customer rating, projects, attendance; each scaled to mean 50, SD 10).

**The question.** How strongly is the attitude side linked to the performance side, and through which measures?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/canonical_correlation.csv` (600 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Engagement,Autonomy,Training completed,Sales score,Customer rating score,Projects score,Attendance score
1.0,0.87,0.91,1.0,52.0,71.5,40.1,58.9
2.0,-0.45,-0.45,0.33,40.4,44.6,40.2,46.9
3.0,0.93,1.26,1.0,67.9,60.7,65.5,56.4
4.0,0.27,-0.25,0.33,46.6,47.0,46.9,48.9
5.0,0.41,1.28,0.67,74.7,40.9,46.4,44.7
6.0,-1.61,-2.05,0.0,31.7,44.8,37.4,41.6
7.0,-0.36,0.2,0.67,69.1,45.2,45.2,50.9
8.0,-0.4,0.41,1.0,49.2,47.7,55.3,71.0
9.0,0.08,-0.36,0.33,35.1,56.4,40.4,56.9
10.0,1.07,-0.66,0.0,48.5,48.2,50.1,61.8
11.0,0.93,2.06,0.0,57.2,44.5,65.1,47.0
12.0,-1.91,-1.35,0.33,37.4,27.7,55.2,31.3
13.0,0.41,0.32,0.0,54.2,57.5,64.6,39.9
14.0,0.45,-0.51,0.33,55.5,45.6,34.8,54.6
15.0,0.8,0.12,0.33,52.4,45.8,48.6,34.8
16.0,-0.14,1.17,0.67,39.0,51.8,55.7,57.0
17.0,-0.38,-0.12,0.67,56.5,48.3,34.2,31.9
18.0,-0.61,0.54,0.67,43.7,47.8,34.0,41.0
19.0,-0.32,0.87,0.67,57.8,49.4,60.7,46.9
20.0,-0.38,-1.02,0.67,45.8,53.8,53.3,73.8
21.0,-0.5,-0.76,0.33,24.6,46.2,37.1,44.9
22.0,-0.06,1.96,0.33,49.8,36.4,65.3,45.8
23.0,0.68,0.27,0.33,55.1,49.1,46.9,67.3
24.0,0.22,-0.93,0.67,46.0,51.2,54.8,68.7
25.0,-0.37,0.95,0.67,48.4,64.1,31.3,38.0
26.0,1.08,0.91,0.67,68.2,34.7,56.4,64.4
27.0,-1.25,-0.54,0.67,44.0,51.6,44.9,46.8
28.0,0.51,0.09,0.33,57.4,44.2,57.0,54.6
29.0,-0.58,-0.56,0.67,44.9,49.7,51.1,42.4
30.0,0.12,0.12,0.33,44.4,47.5,40.3,60.7
31.0,0.6,1.15,1.0,47.2,40.0,42.4,59.1
32.0,0.25,-0.27,0.67,54.7,63.2,49.3,36.8
33.0,0.17,-0.82,0.33,47.1,50.6,58.6,35.5
34.0,0.33,-0.13,1.0,54.7,49.3,54.9,48.2
35.0,-0.56,0.33,0.67,37.7,47.5,42.1,54.3
36.0,0.62,-0.82,0.33,48.4,54.0,48.2,55.5
37.0,0.46,0.47,0.67,50.1,57.5,48.5,69.2
38.0,1.74,0.87,1.0,66.5,69.6,66.3,66.1
39.0,-0.51,-0.38,0.33,47.9,53.9,51.8,61.0
40.0,0.64,0.09,1.0,58.2,64.1,64.3,50.5
41.0,0.42,0.87,0.67,59.4,71.2,64.5,52.8
42.0,-0.89,-0.21,0.0,40.6,54.4,53.0,54.3
43.0,-1.13,-0.65,0.33,57.3,50.7,46.4,41.6
44.0,-0.63,-0.38,0.33,61.7,51.8,53.7,55.0
45.0,0.29,-0.69,0.33,50.5,53.5,30.0,54.1
46.0,0.52,-0.72,0.0,42.0,42.0,49.9,55.2
47.0,0.32,0.24,0.67,52.4,41.3,57.5,42.1
48.0,-0.47,-0.96,0.33,45.4,55.7,66.6,50.6
49.0,0.09,-0.06,0.67,50.4,54.1,30.5,57.7
50.0,-0.32,-0.13,0.0,42.9,44.6,43.6,44.6
51.0,-0.55,0.47,0.33,44.8,62.1,40.3,50.1
52.0,-1.07,-0.62,0.33,30.4,47.9,39.9,22.5
53.0,-0.21,-0.66,0.33,33.1,37.5,47.4,36.1
54.0,0.02,1.5,0.67,46.4,51.6,50.2,32.9
55.0,0.57,0.47,0.67,53.4,44.3,57.0,54.2
56.0,0.86,0.56,0.67,49.9,62.7,56.0,46.2
57.0,-1.09,-1.79,0.0,29.0,54.8,23.2,24.8
58.0,0.2,-0.35,0.67,51.1,57.7,53.6,40.0
59.0,0.6,-1.52,1.0,64.0,52.9,61.1,48.4
60.0,-0.43,0.76,0.0,54.4,46.4,64.7,49.8
61.0,0.08,-0.52,0.33,44.4,29.0,79.6,41.8
62.0,0.02,0.07,0.0,64.4,33.2,47.0,60.5
63.0,-0.26,0.48,0.67,46.5,54.6,36.0,57.2
64.0,0.48,0.48,0.67,49.6,66.2,49.4,46.3
65.0,-0.74,-0.07,0.67,56.6,40.9,57.8,23.5
66.0,-0.08,-0.51,0.67,47.7,48.0,45.2,36.9
67.0,-0.15,-0.67,0.67,51.6,49.6,51.8,54.3
68.0,1.58,0.41,1.0,51.5,72.9,51.7,76.2
69.0,0.08,0.67,0.33,50.0,45.6,34.7,48.4
70.0,-1.16,0.16,0.0,47.9,70.9,41.7,45.8
71.0,0.41,0.26,0.33,33.5,38.4,46.9,35.9
72.0,0.51,-0.49,0.33,50.5,44.9,23.3,43.9
73.0,-0.58,0.32,0.67,46.8,49.1,29.5,51.7
74.0,0.33,0.11,0.33,43.8,39.5,47.7,61.2
75.0,-0.2,-0.07,0.33,44.9,49.6,43.6,52.2
76.0,-0.03,0.48,0.33,46.8,51.3,76.0,59.0
77.0,1.22,-0.57,1.0,43.9,55.3,42.1,52.6
78.0,0.61,0.18,0.33,57.2,57.6,56.3,51.3
79.0,0.39,-0.56,0.33,40.3,42.9,45.8,44.7
80.0,-0.08,0.18,0.33,59.5,47.1,49.8,51.5
81.0,0.43,-0.33,0.67,46.0,39.3,32.6,48.8
82.0,-0.11,-0.43,0.67,45.8,60.5,41.9,40.3
83.0,0.07,1.05,1.0,59.0,54.6,52.0,47.2
84.0,-0.31,0.12,0.0,66.6,58.7,56.9,58.7
85.0,0.65,0.0,0.67,64.9,50.3,50.8,59.8
86.0,0.46,0.11,1.0,51.3,57.7,54.1,55.6
87.0,1.6,1.11,0.67,60.8,67.5,60.8,38.4
88.0,-0.64,-0.84,0.33,43.6,36.9,46.1,43.3
89.0,-0.24,-1.15,0.0,35.2,56.7,45.0,56.6
90.0,0.21,-0.27,0.67,70.4,52.4,44.1,66.2
91.0,-0.36,-0.37,0.67,54.0,65.9,61.7,42.7
92.0,0.42,-0.08,0.67,42.5,55.4,47.9,53.9
93.0,0.15,1.37,1.0,53.5,56.1,54.6,54.4
94.0,0.49,-0.14,1.0,41.9,28.8,49.2,46.6
95.0,-0.79,0.14,0.33,46.6,46.7,53.8,48.9
96.0,0.43,2.22,1.0,61.3,69.1,80.2,33.1
97.0,-0.35,0.26,0.0,43.8,65.0,44.7,48.4
98.0,-1.95,-0.59,0.33,44.0,40.2,37.5,47.8
99.0,1.45,1.7,1.0,65.2,48.2,78.2,74.9
100.0,0.27,0.59,0.33,53.9,27.5,52.1,51.8
101.0,0.32,0.77,0.33,56.2,55.6,47.5,42.4
102.0,0.94,0.45,0.67,44.0,43.7,47.1,47.7
103.0,-1.28,-0.54,0.67,58.3,56.7,65.8,53.0
104.0,1.01,0.49,1.0,63.6,56.6,47.3,69.0
105.0,-0.14,-0.73,0.67,45.4,55.9,31.2,37.2
106.0,0.2,-0.52,0.67,39.0,50.5,45.7,55.3
107.0,-0.39,-0.68,0.67,40.7,57.2,40.0,42.4
108.0,1.08,0.83,0.67,53.6,53.8,30.1,50.5
109.0,-0.56,0.71,0.33,44.8,45.5,46.7,64.7
110.0,0.91,0.92,0.67,56.9,54.7,49.9,52.9
111.0,0.33,1.34,0.33,57.3,46.4,36.6,51.9
112.0,0.0,-0.36,0.33,46.2,49.9,35.2,57.0
113.0,0.07,0.8,0.0,56.4,52.3,44.6,70.1
114.0,0.4,-0.43,0.0,37.5,51.8,62.4,48.9
115.0,0.24,-0.19,0.33,67.5,53.5,43.7,58.1
116.0,-1.01,-1.84,0.33,45.6,45.0,41.9,39.4
117.0,0.61,0.33,0.67,45.2,35.5,49.5,71.0
118.0,-0.49,0.58,0.33,40.8,49.9,63.8,49.2
119.0,2.01,0.71,1.0,52.5,57.5,56.5,58.9
120.0,0.88,1.01,0.67,63.5,55.0,53.3,69.6
121.0,0.93,1.46,1.0,53.7,44.7,48.9,47.5
122.0,1.09,0.31,0.67,53.4,48.6,51.5,48.0
123.0,0.63,-0.36,0.33,57.4,56.9,43.1,54.2
124.0,0.06,-0.18,0.33,38.5,48.1,42.3,42.7
125.0,-0.54,-0.22,0.0,45.0,59.6,53.8,33.2
126.0,0.53,0.52,0.33,67.0,56.2,53.4,55.5
127.0,-0.88,-0.65,0.67,39.4,35.6,48.5,41.9
128.0,0.46,-0.49,0.67,59.0,35.5,46.1,54.7
129.0,-1.36,-0.58,0.33,28.4,49.3,47.5,42.5
130.0,-0.63,-1.02,0.33,49.9,43.4,42.9,41.8
131.0,-0.42,-0.79,1.0,37.4,43.0,49.3,60.8
132.0,0.45,-0.13,0.67,46.1,41.5,50.8,63.0
133.0,-0.89,-0.13,0.67,39.4,48.4,60.5,69.2
134.0,0.26,0.63,0.0,61.1,45.3,51.0,45.1
135.0,0.57,-0.04,0.0,46.2,54.0,46.5,47.1
136.0,-0.33,-0.53,0.0,34.7,25.5,46.5,59.3
137.0,1.38,1.43,1.0,53.3,66.4,48.9,56.9
138.0,-0.28,-1.0,0.33,64.2,45.0,52.0,48.3
139.0,0.71,0.3,0.67,38.4,51.1,52.7,52.1
140.0,-0.18,-1.0,1.0,23.7,41.7,36.6,29.3
141.0,-1.22,1.11,0.0,31.3,36.8,49.1,35.2
142.0,0.09,-0.44,0.33,51.3,55.2,50.5,56.6
143.0,-0.07,0.84,0.0,40.1,41.6,61.2,57.5
144.0,0.08,-0.01,0.67,47.4,47.1,58.7,40.3
145.0,0.18,-0.28,0.33,42.8,44.0,58.3,51.7
146.0,0.73,-0.05,0.67,53.3,59.1,58.4,64.9
147.0,-0.06,0.62,0.67,68.0,50.8,50.9,50.0
148.0,0.98,-0.36,1.0,47.1,39.9,41.1,42.6
149.0,-0.64,-0.91,0.33,40.9,44.2,37.2,54.5
150.0,0.24,0.88,1.0,53.2,62.1,49.8,47.9
151.0,0.15,0.63,0.67,49.7,57.8,42.8,77.4
152.0,-1.3,-0.36,0.33,59.3,40.5,50.9,39.6
153.0,-0.41,-1.36,0.0,56.4,48.9,48.1,45.9
154.0,0.6,1.4,0.33,59.5,63.1,54.4,45.7
155.0,1.22,0.25,0.67,53.2,61.4,68.5,76.1
156.0,0.69,0.1,0.67,61.7,59.9,51.6,56.5
157.0,-0.23,-0.78,0.67,36.1,54.3,55.6,49.9
158.0,0.23,-0.96,0.33,63.4,37.5,48.6,53.9
159.0,-0.14,0.4,0.0,51.4,49.3,64.0,41.9
160.0,0.89,-0.54,0.67,41.2,54.8,44.9,54.9
161.0,-0.46,-0.85,0.33,60.4,55.5,34.5,54.7
162.0,-1.13,0.68,1.0,53.4,55.2,58.0,58.4
163.0,0.71,1.5,0.67,63.6,48.3,53.7,47.4
164.0,-1.03,-0.56,0.33,32.8,61.5,31.6,49.3
165.0,0.39,0.14,0.67,54.3,42.5,48.6,46.8
166.0,-0.59,0.5,0.0,45.6,48.3,59.9,59.5
167.0,1.03,-0.33,0.33,48.2,53.7,59.8,47.2
168.0,-0.35,-0.57,0.67,49.4,29.7,56.4,46.2
169.0,0.54,-0.31,0.0,62.6,34.5,45.9,67.5
170.0,0.48,0.62,1.0,59.1,47.3,44.0,56.7
171.0,0.32,0.37,0.67,59.3,42.3,52.0,40.9
172.0,1.44,0.49,0.67,72.0,64.7,63.1,53.7
173.0,-1.26,-1.39,0.67,48.7,48.5,37.9,21.1
174.0,0.52,0.02,0.33,50.2,33.6,41.4,48.3
175.0,-0.48,-0.6,0.33,36.3,38.1,45.0,45.2
176.0,0.51,-0.25,0.67,53.8,54.2,58.1,46.1
177.0,-0.87,-0.16,0.0,41.9,60.1,44.7,35.4
178.0,-0.04,-0.08,0.33,40.1,53.4,38.8,57.3
179.0,-0.24,-0.27,0.33,41.7,43.1,32.6,42.9
180.0,-0.48,0.12,0.33,59.7,42.3,56.4,30.4
181.0,-0.47,0.17,0.67,58.7,54.1,58.5,51.7
182.0,-0.2,-0.44,1.0,34.2,58.0,39.3,63.8
183.0,-0.75,-0.46,0.33,39.2,53.6,60.7,55.8
184.0,1.69,1.3,1.0,59.5,69.4,67.8,55.6
185.0,-0.32,-1.8,0.67,41.2,35.3,40.4,35.9
186.0,-1.05,-0.49,0.0,37.5,38.4,47.3,47.4
187.0,1.12,0.05,1.0,62.6,50.8,64.8,49.5
188.0,-1.06,0.58,0.33,41.0,48.3,51.8,48.6
189.0,-0.24,1.02,0.67,56.6,41.4,49.6,56.3
190.0,0.92,0.41,0.33,57.6,52.3,71.0,54.7
191.0,-0.63,-0.84,0.33,49.4,64.2,28.4,56.0
192.0,-0.98,-0.81,0.0,34.9,45.6,45.4,40.7
193.0,-0.29,1.1,0.67,45.1,63.9,44.3,54.9
194.0,-0.82,-0.94,0.0,46.4,53.2,35.3,51.6
195.0,0.82,0.29,0.67,65.3,47.8,47.4,61.3
196.0,0.79,0.28,0.33,56.5,55.1,54.0,49.2
197.0,-1.22,-0.09,0.67,46.3,39.1,48.8,46.0
198.0,-0.06,-1.02,0.33,55.6,42.7,43.2,42.8
199.0,1.04,0.27,0.33,49.2,51.5,58.2,50.2
200.0,-2.71,-0.35,0.0,28.6,31.5,33.2,44.0
201.0,-1.44,0.02,0.67,38.5,39.4,40.3,42.5
202.0,-0.87,-0.29,0.33,47.1,37.9,52.5,36.5
203.0,-0.81,-1.14,0.0,54.6,53.0,43.6,52.1
204.0,0.37,-0.73,0.33,60.4,43.7,61.7,34.3
205.0,-1.26,-0.64,0.0,39.8,60.2,42.3,39.9
206.0,-0.23,-0.41,0.0,37.3,35.1,49.6,44.9
207.0,-1.16,-0.63,1.0,47.0,44.6,44.4,46.6
208.0,-0.92,-0.84,0.33,42.3,49.6,46.2,37.2
209.0,-0.83,-0.57,0.0,43.6,30.2,39.5,41.1
210.0,0.45,0.21,0.33,46.2,55.1,45.6,48.6
211.0,-0.63,1.3,0.67,60.5,49.9,53.0,38.0
212.0,-0.88,-1.01,0.0,33.1,43.1,30.0,51.5
213.0,-0.67,-1.04,0.33,59.5,37.3,44.8,61.7
214.0,0.47,0.75,1.0,60.2,40.0,75.0,63.7
215.0,1.37,0.9,1.0,71.4,52.7,52.2,68.3
216.0,-0.11,-0.42,0.33,54.7,39.2,47.9,52.5
217.0,-0.81,0.2,0.33,50.5,62.9,52.1,66.2
218.0,0.28,-0.5,0.33,44.5,34.6,44.3,39.9
219.0,-0.22,-0.01,0.33,48.9,36.2,43.4,43.9
220.0,-1.12,0.61,0.33,43.0,46.0,50.6,52.9
221.0,-1.0,-0.82,0.67,27.3,36.0,45.9,50.6
222.0,-0.98,-1.31,0.33,48.3,44.8,71.5,42.3
223.0,0.15,0.15,0.33,51.3,52.9,61.8,41.4
224.0,-0.01,0.09,1.0,56.2,54.2,67.5,56.4
225.0,-0.1,-0.7,1.0,65.0,55.2,56.0,39.4
226.0,-0.29,0.31,0.67,55.3,59.0,53.7,54.5
227.0,0.63,0.64,0.67,64.3,65.6,36.6,56.6
228.0,0.47,1.41,0.33,60.4,51.8,70.2,46.4
229.0,-0.79,-1.23,0.0,42.8,42.9,45.1,46.4
230.0,-0.03,-0.1,0.0,44.3,51.4,43.8,51.0
231.0,-1.13,-1.79,0.33,31.1,43.3,45.3,53.1
232.0,-0.41,-0.02,0.67,49.8,37.1,69.2,39.1
233.0,1.26,0.16,0.67,54.4,67.7,58.0,54.4
234.0,0.72,0.49,0.67,51.9,65.7,39.7,71.2
235.0,1.09,0.5,0.67,68.7,63.1,50.1,46.0
236.0,-0.17,1.25,0.67,66.4,61.9,59.1,56.2
237.0,0.55,-0.49,1.0,58.2,64.5,41.1,62.1
238.0,-0.04,-0.35,0.33,42.2,61.8,55.7,40.9
239.0,-0.68,-0.3,0.33,45.2,51.9,55.8,38.9
240.0,-0.56,0.41,0.67,45.9,34.4,55.9,48.6
241.0,-0.58,0.31,0.0,44.1,45.8,49.4,45.4
242.0,-0.21,-0.39,0.33,44.8,51.9,54.7,49.2
243.0,-0.53,-0.75,0.0,40.5,41.7,54.3,53.8
244.0,0.13,-0.23,0.67,55.5,45.9,29.2,56.0
245.0,-0.32,-0.43,0.33,41.1,29.0,43.3,52.6
246.0,-1.01,-0.76,0.67,38.2,51.0,41.1,51.8
247.0,-0.47,-1.18,0.33,67.2,45.1,48.4,34.1
248.0,-0.68,0.43,1.0,46.8,35.0,62.3,58.7
249.0,0.62,0.78,0.67,48.0,44.2,51.1,40.9
250.0,-0.39,0.27,0.0,44.8,39.3,63.9,45.0
251.0,0.17,0.5,0.67,55.0,53.0,62.0,42.9
252.0,-0.22,-0.72,0.67,52.9,50.7,28.0,33.7
253.0,0.4,0.01,1.0,58.2,53.4,58.7,70.2
254.0,0.12,-0.59,0.33,50.5,50.8,62.9,41.3
255.0,-1.19,-0.93,0.33,51.4,47.1,31.9,58.3
256.0,0.08,0.34,0.67,46.5,43.5,42.4,56.2
257.0,0.88,1.0,0.67,67.7,54.2,51.8,59.6
258.0,0.13,-0.76,0.33,48.9,50.1,38.8,62.4
259.0,-0.18,-0.33,0.33,39.0,43.8,39.6,42.3
260.0,0.13,-0.09,1.0,45.8,57.0,44.4,66.0
261.0,-0.9,0.52,0.0,55.4,52.1,57.0,51.0
262.0,1.44,0.96,1.0,72.5,58.7,51.7,52.7
263.0,0.79,1.01,1.0,64.4,61.5,52.8,58.6
264.0,-0.44,-0.33,0.33,50.9,46.9,60.5,53.0
265.0,0.13,0.61,0.67,43.1,40.1,45.2,46.5
266.0,-1.07,-0.18,0.0,38.4,57.9,52.6,61.7
267.0,-0.74,-0.53,0.33,57.5,37.6,45.3,41.5
268.0,0.5,0.45,0.67,58.4,52.8,64.5,49.9
269.0,0.15,1.78,0.67,66.8,53.0,52.6,62.6
270.0,0.3,-1.66,0.33,44.9,51.5,50.5,34.1
271.0,1.47,0.83,0.33,56.9,44.4,56.0,44.1
272.0,-0.21,0.09,0.67,38.5,61.1,53.0,53.0
273.0,-0.6,0.52,0.67,45.4,66.0,43.0,36.4
274.0,-0.99,-0.57,0.67,41.5,39.4,44.1,50.2
275.0,-0.21,-1.11,1.0,46.3,51.9,35.0,54.2
276.0,0.04,-0.56,0.33,55.4,40.5,46.9,52.6
277.0,0.22,0.34,0.0,67.3,56.4,34.9,47.9
278.0,-0.26,-0.61,0.0,38.9,51.7,35.8,65.0
279.0,0.03,-0.67,0.67,37.0,52.3,29.7,38.5
280.0,0.27,0.54,0.33,55.6,53.5,58.2,52.5
281.0,-1.02,-1.58,0.33,31.3,37.2,52.7,23.5
282.0,0.32,0.87,0.67,34.4,51.3,34.3,29.9
283.0,-0.5,-0.37,1.0,45.5,58.1,62.3,54.8
284.0,1.19,0.4,1.0,54.8,47.2,51.2,62.1
285.0,0.37,-0.09,1.0,48.9,53.9,59.1,29.5
286.0,1.15,0.71,0.67,58.2,76.8,60.3,57.2
287.0,0.76,0.12,0.33,37.3,55.6,35.5,53.7
288.0,-1.38,-1.0,0.33,46.6,36.1,43.1,53.0
289.0,0.97,-0.4,1.0,51.3,56.7,54.5,55.2
290.0,0.06,0.78,0.67,51.4,48.4,54.9,49.2
291.0,0.23,0.47,0.0,52.4,56.8,56.7,49.9
292.0,-1.26,-1.37,0.33,54.4,56.5,48.5,41.0
293.0,0.23,0.33,0.33,45.2,65.1,58.9,37.0
294.0,-0.47,-1.29,0.33,34.2,37.6,42.7,41.9
295.0,-0.54,-0.72,0.0,45.3,40.1,44.3,43.3
296.0,-0.01,0.76,0.33,63.6,61.1,50.0,60.4
297.0,-0.71,-0.14,0.33,46.6,53.7,52.8,58.7
298.0,-0.84,-0.08,0.0,58.3,47.1,51.6,47.1
299.0,0.24,0.55,0.33,53.9,47.6,69.6,36.8
300.0,1.73,0.46,1.0,61.3,54.3,51.3,70.3
301.0,-1.47,0.32,0.33,36.3,36.7,57.0,53.2
302.0,-0.39,-0.64,0.33,42.3,49.5,32.9,42.5
303.0,-0.06,-0.68,0.67,38.5,42.1,49.8,49.0
304.0,0.6,0.77,0.67,73.0,42.1,73.9,71.5
305.0,-0.12,0.94,0.67,38.0,46.2,43.6,59.2
306.0,0.2,-0.14,0.33,42.8,49.4,59.0,49.5
307.0,0.06,-0.16,0.0,48.5,47.8,48.2,51.3
308.0,-0.77,-0.9,0.33,34.7,41.5,39.9,66.0
309.0,-0.16,-0.08,1.0,55.7,51.1,49.3,53.7
310.0,0.26,-0.67,1.0,66.4,63.7,38.9,37.1
311.0,0.51,0.76,1.0,53.9,44.9,48.1,41.4
312.0,0.12,0.58,0.0,46.3,55.8,43.0,59.4
313.0,0.2,0.76,0.0,56.0,54.3,49.8,43.1
314.0,-0.08,-0.13,0.67,51.1,62.3,51.9,38.8
315.0,-1.16,-0.31,0.0,39.7,52.3,40.2,55.4
316.0,-0.3,-0.24,0.67,49.4,43.9,45.3,41.4
317.0,-0.41,0.18,0.67,42.5,41.1,54.0,31.7
318.0,0.16,0.11,1.0,55.7,48.7,46.6,51.1
319.0,-1.35,-1.07,0.33,45.4,24.6,51.1,57.1
320.0,-0.38,0.68,0.33,49.6,57.7,48.0,55.0
321.0,1.14,0.05,0.67,68.1,53.0,45.2,49.8
322.0,-1.12,0.85,0.67,38.1,50.4,21.4,53.4
323.0,-0.03,-0.75,0.0,46.0,36.8,64.8,55.2
324.0,-0.45,-0.35,0.67,69.3,44.2,63.2,45.8
325.0,0.3,-1.13,1.0,56.7,48.2,48.0,46.9
326.0,-0.22,0.49,0.33,49.7,48.7,49.9,47.0
327.0,0.66,1.01,0.67,49.1,43.9,54.0,49.1
328.0,-0.43,-0.6,0.33,42.0,31.8,36.8,41.7
329.0,-0.51,0.81,0.0,51.8,61.0,59.7,46.0
330.0,0.01,0.44,0.67,45.7,56.0,55.0,48.5
331.0,0.55,0.02,0.67,46.7,52.1,50.6,68.4
332.0,0.51,0.21,0.67,56.2,65.3,67.3,53.0
333.0,-0.16,0.28,0.33,41.7,43.4,52.0,65.1
334.0,0.38,0.15,0.33,39.0,45.4,47.7,52.6
335.0,-0.14,-0.26,0.33,56.4,50.4,60.6,56.5
336.0,-0.44,-0.7,0.0,48.4,33.1,40.5,37.5
337.0,-0.75,-0.62,0.33,29.5,33.9,36.5,39.5
338.0,-0.2,0.26,0.33,48.9,45.5,52.7,43.2
339.0,-0.15,0.38,1.0,42.9,52.3,51.4,41.9
340.0,-2.45,-0.86,0.0,34.0,27.1,48.8,14.0
341.0,-0.52,0.29,1.0,59.2,36.4,54.6,37.7
342.0,-0.33,-1.34,0.33,39.9,43.0,35.7,42.0
343.0,0.08,-0.47,0.33,58.2,48.2,37.9,50.0
344.0,-0.46,0.22,0.33,51.7,61.6,60.4,46.7
345.0,-0.25,0.34,0.0,55.3,51.4,41.5,50.0
346.0,0.11,0.99,0.67,55.9,41.6,56.6,43.4
347.0,0.55,-0.41,0.67,41.4,50.5,40.6,55.0
348.0,0.88,0.03,0.67,51.0,39.7,48.0,69.0
349.0,-0.62,-0.72,0.0,32.3,44.3,49.5,45.0
350.0,0.53,1.34,0.33,54.1,64.4,62.7,46.0
351.0,0.2,0.25,0.67,48.3,49.3,63.6,62.2
352.0,-0.3,-0.82,0.0,37.5,65.5,50.7,40.2
353.0,0.43,-0.31,0.67,66.9,58.9,56.2,59.2
354.0,0.67,1.26,0.33,61.2,50.3,50.2,50.5
355.0,-0.33,0.78,0.33,43.6,43.3,48.9,46.8
356.0,0.24,0.19,0.67,55.7,45.5,28.3,48.8
357.0,-1.03,-1.16,0.0,56.9,42.7,61.2,50.1
358.0,-0.09,0.0,1.0,39.4,51.5,45.0,51.4
359.0,-0.24,-1.2,0.33,46.7,38.9,50.5,36.7
360.0,0.74,-0.12,0.33,75.7,55.8,62.3,57.9
361.0,0.5,0.07,1.0,64.3,40.4,64.0,66.0
362.0,1.46,0.96,0.33,54.2,59.6,43.4,63.5
363.0,-0.18,-0.15,0.0,47.0,39.8,51.0,40.1
364.0,-0.41,-0.71,0.33,43.0,60.4,48.1,47.1
365.0,-0.07,0.44,0.67,64.7,47.0,48.7,76.5
366.0,0.35,-0.81,0.33,42.5,42.8,48.3,33.5
367.0,0.13,-0.06,0.67,52.6,59.9,49.2,44.6
368.0,-0.51,-1.29,0.33,47.7,24.4,47.4,50.7
369.0,-0.15,-0.74,1.0,54.2,69.9,43.7,41.7
370.0,0.61,1.56,0.33,54.1,73.2,61.3,44.8
371.0,1.12,-0.14,0.67,46.4,47.3,42.0,54.8
372.0,0.11,0.31,0.33,49.1,58.3,41.0,46.5
373.0,-0.17,0.67,0.67,67.8,56.2,69.5,41.7
374.0,-0.11,0.24,0.67,43.5,60.6,63.2,39.9
375.0,0.33,-0.12,0.67,51.2,44.1,38.4,42.9
376.0,-1.5,-2.28,0.33,51.0,47.7,48.2,47.1
377.0,-1.34,-0.05,0.33,54.9,42.4,51.0,52.7
378.0,-0.36,0.24,0.67,55.2,40.3,32.7,36.7
379.0,0.65,0.49,0.33,49.5,50.6,67.8,61.4
380.0,-0.66,-0.29,0.0,48.8,30.8,56.3,49.3
381.0,1.1,0.3,1.0,48.1,59.4,47.5,59.5
382.0,-0.63,-0.45,0.33,42.7,45.6,61.3,42.5
383.0,-1.13,0.3,0.33,50.9,38.5,38.8,47.3
384.0,-0.92,-1.06,0.0,51.8,48.6,30.8,33.9
385.0,0.84,-0.06,1.0,51.1,55.6,47.9,41.4
386.0,-1.01,0.17,0.33,50.2,64.2,46.5,46.2
387.0,-0.15,0.01,0.67,43.5,45.6,45.1,29.2
388.0,2.03,0.18,1.0,63.3,56.0,69.4,68.1
389.0,0.86,-0.47,0.67,47.7,37.0,38.0,38.4
390.0,-0.85,-0.43,0.67,41.7,46.1,46.5,58.2
391.0,0.44,0.26,0.33,81.8,55.6,28.1,64.8
392.0,0.17,-0.07,0.33,43.3,42.6,46.5,48.6
393.0,-0.48,0.04,1.0,56.1,54.5,52.1,62.8
394.0,0.11,0.06,0.33,47.4,44.3,47.7,50.5
395.0,-0.18,-1.14,0.33,68.7,43.0,36.8,57.0
396.0,-1.18,-0.41,0.33,33.6,52.3,45.4,48.6
397.0,0.12,-0.73,0.67,44.6,49.8,42.9,49.7
398.0,1.42,-0.38,0.33,52.7,50.9,50.4,48.6
399.0,0.22,-0.63,0.67,66.7,54.1,58.5,51.6
400.0,0.56,0.44,1.0,54.0,53.1,60.8,61.7
401.0,-1.27,-0.78,0.33,43.8,30.1,50.2,38.8
402.0,0.21,0.02,0.33,42.6,62.9,50.7,52.0
403.0,-1.21,-0.37,0.67,60.9,54.4,49.9,47.5
404.0,0.25,0.13,0.33,44.3,52.3,45.3,49.4
405.0,0.9,1.93,1.0,55.9,49.5,70.7,64.0
406.0,-0.02,1.31,0.33,61.2,44.1,72.8,49.3
407.0,-0.5,0.29,0.0,44.6,53.7,43.0,54.1
408.0,-0.78,0.16,0.0,24.4,43.1,47.6,51.1
409.0,0.23,-0.1,0.33,48.1,61.5,52.5,51.1
410.0,0.54,0.22,0.67,51.8,40.5,66.1,42.2
411.0,0.55,0.14,0.67,52.8,51.8,49.9,41.6
412.0,-0.35,1.07,0.33,61.3,40.4,55.8,48.2
413.0,-0.13,0.53,0.33,55.6,49.2,42.0,67.9
414.0,1.2,0.06,1.0,54.0,56.7,46.3,47.3
415.0,-0.15,-0.74,0.33,37.8,40.2,51.3,49.1
416.0,1.43,0.42,1.0,68.8,63.7,46.6,71.1
417.0,-0.71,0.03,0.67,33.3,42.3,37.4,39.0
418.0,0.99,0.27,0.33,50.2,60.2,52.6,38.1
419.0,-0.31,-0.16,0.33,47.7,58.0,52.0,77.1
420.0,-0.25,-0.39,0.33,45.8,59.4,62.6,59.2
421.0,-0.23,0.19,0.33,34.0,57.3,41.2,52.6
422.0,1.56,1.29,0.33,53.7,67.1,49.0,52.4
423.0,-1.68,-0.84,0.0,34.0,41.7,31.9,27.8
424.0,0.35,-0.18,0.67,48.1,53.3,43.6,42.9
425.0,-0.54,-0.41,0.33,46.1,61.1,48.4,38.6
426.0,-0.63,-0.55,0.0,37.5,36.6,51.3,49.7
427.0,-1.1,0.37,0.0,37.6,46.9,32.7,58.4
428.0,0.17,-0.03,0.33,45.3,39.3,47.0,40.7
429.0,0.67,0.59,0.67,52.5,47.2,50.6,64.8
430.0,1.43,0.97,1.0,59.5,64.2,38.0,85.3
431.0,-0.56,-0.7,0.33,41.9,38.1,53.0,58.6
432.0,-0.01,-0.65,1.0,41.5,41.6,51.9,53.5
433.0,0.8,-1.1,0.67,37.0,58.5,47.1,52.0
434.0,-0.67,0.13,0.67,44.5,59.6,43.7,56.1
435.0,-0.88,-0.97,0.0,37.6,31.6,53.9,40.5
436.0,1.07,0.64,0.67,36.5,48.0,49.0,44.5
437.0,1.43,0.84,0.33,57.1,49.9,61.2,50.0
438.0,0.83,1.14,0.67,63.6,55.4,66.1,35.9
439.0,0.97,-0.89,1.0,70.2,67.3,37.5,48.3
440.0,0.18,-0.84,0.33,56.5,34.7,48.4,41.7
441.0,0.59,0.19,0.33,49.8,38.6,39.2,62.5
442.0,-0.76,-0.67,0.33,31.7,63.2,47.1,44.1
443.0,-0.56,0.45,0.33,48.7,47.1,46.1,51.0
444.0,0.04,-1.45,0.33,38.8,51.5,31.2,52.6
445.0,-0.13,-0.06,0.67,55.7,52.1,44.9,49.9
446.0,-0.13,-0.35,0.67,53.1,51.0,62.0,48.9
447.0,-0.91,1.51,0.33,59.7,67.3,59.0,60.8
448.0,-0.4,0.75,0.67,52.8,37.5,43.8,46.1
449.0,-0.85,-1.67,0.0,42.5,55.6,45.3,37.2
450.0,-0.1,-0.2,0.67,65.9,44.5,41.4,47.6
451.0,-1.34,-0.85,0.0,36.6,27.1,30.8,39.8
452.0,-1.3,-0.91,0.33,49.5,42.9,50.2,37.0
453.0,1.44,0.03,0.67,55.0,64.4,56.2,64.7
454.0,-0.57,0.58,0.67,50.2,25.9,66.0,49.5
455.0,0.04,-0.16,0.67,71.9,57.5,23.2,49.5
456.0,-2.06,-0.7,0.0,60.5,41.2,44.0,45.5
457.0,-0.38,0.66,0.33,45.9,42.9,46.1,49.9
458.0,-0.12,0.02,0.67,55.8,38.6,47.7,53.7
459.0,1.44,-0.16,0.33,53.2,67.6,48.9,38.2
460.0,0.06,-0.7,0.0,37.0,40.3,46.7,44.7
461.0,1.39,1.36,1.0,58.0,53.2,54.8,46.9
462.0,-1.14,0.51,0.67,42.0,54.3,47.8,45.9
463.0,0.41,-0.15,0.67,50.0,55.3,45.5,53.1
464.0,-0.81,0.45,0.33,63.3,41.6,49.9,52.0
465.0,-0.62,0.36,0.33,67.3,59.1,57.6,50.6
466.0,0.49,-0.28,0.0,51.2,45.1,43.9,42.2
467.0,0.56,-0.91,1.0,48.3,60.6,42.8,37.9
468.0,0.16,0.79,0.67,43.6,59.4,65.6,50.7
469.0,0.14,-0.21,0.33,39.1,45.3,22.4,40.0
470.0,0.26,-1.2,0.33,53.6,59.9,48.3,55.4
471.0,0.4,-0.84,0.67,53.7,70.0,58.1,50.2
472.0,0.34,0.77,0.33,58.4,56.9,62.7,53.8
473.0,-0.13,0.97,0.67,69.3,49.0,50.1,58.1
474.0,0.55,1.3,0.33,45.1,52.0,54.4,49.9
475.0,0.66,-0.26,0.67,67.4,47.0,41.1,62.6
476.0,-0.13,-0.06,0.33,44.2,62.0,56.9,63.7
477.0,-1.35,0.54,0.0,45.4,38.0,52.9,39.7
478.0,0.2,0.24,0.33,30.2,48.0,50.0,54.4
479.0,-0.6,-0.54,0.33,53.2,55.9,47.3,41.6
480.0,0.55,0.41,0.33,55.2,39.6,65.4,55.5
481.0,-0.38,-1.54,0.67,44.8,42.4,39.3,61.7
482.0,-0.48,-0.11,0.67,46.3,37.7,44.1,45.9
483.0,0.65,-0.2,0.0,59.1,50.5,27.3,67.7
484.0,-0.78,-1.16,0.33,66.1,51.6,48.0,38.3
485.0,0.36,-0.57,0.0,50.3,61.5,47.7,47.2
486.0,1.0,0.44,1.0,59.7,56.5,32.3,51.7
487.0,-0.54,0.25,0.0,59.7,37.3,53.7,59.2
488.0,0.23,-1.3,0.33,30.3,33.0,33.1,50.4
489.0,-1.7,-0.87,0.0,37.9,39.2,48.9,52.0
490.0,0.69,0.24,1.0,28.0,54.8,43.7,48.0
491.0,0.68,-1.26,0.33,55.6,40.0,34.5,48.7
492.0,-0.39,0.89,0.0,49.0,45.1,43.0,45.7
493.0,0.06,-0.26,0.0,56.3,52.2,37.3,59.5
494.0,0.3,0.71,0.33,41.5,45.0,50.3,46.2
495.0,0.06,0.37,0.33,54.5,45.4,52.3,48.8
496.0,-0.88,-0.63,0.67,40.7,66.0,32.9,42.6
497.0,0.21,0.44,0.33,46.8,56.4,42.9,37.1
498.0,-0.91,0.0,0.67,32.0,44.0,37.2,46.7
499.0,0.25,2.11,0.33,51.5,46.9,63.1,42.9
500.0,0.71,0.13,1.0,53.0,47.8,66.8,60.5
501.0,-0.54,-0.37,0.0,34.0,39.2,56.3,45.4
502.0,0.54,-0.33,0.33,63.5,54.0,20.8,41.7
503.0,0.38,0.72,0.67,39.7,39.0,57.5,59.3
504.0,-0.02,-1.2,0.33,60.3,38.6,66.3,47.7
505.0,-1.28,0.29,0.0,44.3,47.4,55.5,41.5
506.0,-0.44,-0.21,0.33,41.9,49.6,49.8,36.7
507.0,0.59,-0.43,0.33,46.3,59.3,48.7,45.0
508.0,0.49,1.75,0.67,57.9,33.9,36.4,52.1
509.0,1.16,-0.16,0.33,44.6,38.9,37.5,39.1
510.0,-0.23,-0.21,0.33,52.6,56.1,47.8,52.0
511.0,1.03,0.58,0.33,58.1,45.0,47.1,46.2
512.0,-0.78,-0.77,0.33,44.1,57.2,66.0,44.6
513.0,-0.97,0.58,0.33,39.1,36.7,55.2,49.4
514.0,-0.49,-0.8,0.67,48.4,30.1,55.4,37.3
515.0,-0.3,-1.18,0.67,57.3,54.9,30.8,55.1
516.0,-0.73,0.16,0.0,44.3,49.9,49.6,48.8
517.0,0.38,0.83,1.0,63.8,50.9,37.6,64.4
518.0,0.86,0.56,0.67,62.6,69.6,51.7,60.4
519.0,-0.29,-0.76,0.0,41.4,34.9,48.4,60.9
520.0,-0.75,-1.07,0.0,33.4,44.3,40.3,52.3
521.0,-1.6,-0.37,0.67,47.8,39.4,60.7,63.2
522.0,1.34,0.92,0.33,64.9,54.5,52.0,51.8
523.0,0.56,0.53,0.33,60.6,43.7,52.6,59.3
524.0,-0.63,-1.65,0.33,41.7,42.8,41.0,29.5
525.0,-0.11,0.04,0.67,47.4,54.8,40.9,62.0
526.0,-0.47,0.01,0.67,42.3,36.9,40.1,53.4
527.0,0.03,-0.18,0.67,49.5,40.0,53.3,56.1
528.0,-0.91,-1.57,0.0,34.6,48.5,40.4,26.1
529.0,-0.03,0.44,0.0,53.5,42.3,55.9,51.1
530.0,-0.85,-0.77,1.0,45.7,48.2,54.1,52.6
531.0,0.16,0.56,0.67,45.9,56.2,47.2,44.7
532.0,-2.06,0.52,0.0,42.5,57.8,46.0,26.7
533.0,-0.11,0.36,0.33,42.9,59.9,44.5,43.2
534.0,0.82,1.1,1.0,54.5,47.3,52.4,64.5
535.0,0.81,0.57,0.67,66.3,59.5,46.9,34.8
536.0,0.03,0.27,1.0,41.4,58.1,55.8,55.6
537.0,0.23,-1.02,0.67,39.8,46.6,32.0,56.3
538.0,-0.64,-1.04,0.67,40.9,47.9,53.0,49.4
539.0,-0.72,-0.78,0.67,51.2,35.4,71.3,24.0
540.0,0.31,-1.24,0.33,40.5,29.1,42.2,57.4
541.0,-0.61,0.88,0.33,37.6,51.7,61.7,56.8
542.0,-0.82,0.17,0.33,58.0,34.2,47.8,45.8
543.0,-0.95,-1.04,0.33,55.4,44.8,50.0,55.4
544.0,1.69,1.04,0.67,64.2,47.3,59.1,46.9
545.0,0.66,0.49,0.67,53.9,48.8,59.0,47.9
546.0,0.54,-0.17,0.33,50.2,62.6,64.9,38.6
547.0,-1.87,0.45,0.67,51.5,32.4,60.9,36.5
548.0,0.28,1.05,0.33,55.4,36.1,48.8,48.8
549.0,-0.88,-0.84,0.67,55.1,26.3,50.4,47.0
550.0,1.03,-0.54,0.67,43.7,59.8,54.4,49.4
551.0,-0.22,-0.89,0.67,35.6,63.7,41.8,48.2
552.0,-0.48,-0.53,0.67,37.7,49.4,53.6,44.2
553.0,-0.98,0.59,0.67,39.7,31.4,57.4,52.0
554.0,-0.53,0.4,0.33,37.8,56.0,43.4,46.8
555.0,-1.04,-0.88,0.0,36.5,28.6,36.5,51.0
556.0,0.36,0.72,1.0,52.1,46.6,53.7,47.9
557.0,-0.95,-0.75,0.33,56.4,45.1,48.5,47.8
558.0,0.53,1.88,0.67,64.1,62.7,67.0,63.7
559.0,0.44,-0.07,0.67,56.7,58.4,63.0,52.2
560.0,0.06,-0.88,0.67,52.6,73.5,52.5,63.4
561.0,0.46,0.93,0.33,44.4,55.9,52.7,53.7
562.0,0.77,-0.3,0.0,49.5,50.8,40.9,62.1
563.0,0.04,0.86,0.67,49.0,60.6,54.2,63.6
564.0,-0.63,0.68,0.67,42.8,42.8,48.2,60.1
565.0,-1.75,-0.43,0.0,46.5,49.4,58.9,36.9
566.0,0.52,0.16,0.33,56.1,55.5,43.2,45.3
567.0,-0.79,-0.31,1.0,51.1,59.0,35.1,36.2
568.0,0.6,-0.09,0.67,61.4,64.7,52.7,69.6
569.0,0.04,0.94,0.33,54.9,60.8,49.7,55.1
570.0,-0.17,-0.03,1.0,42.5,54.0,54.8,54.5
571.0,-0.05,-0.81,0.0,52.8,46.8,45.7,53.1
572.0,0.53,-0.03,0.33,52.7,48.0,63.2,40.7
573.0,0.82,-0.2,0.33,34.0,24.6,57.7,44.1
574.0,0.49,1.01,0.33,41.5,55.0,52.2,56.3
575.0,0.24,1.3,1.0,76.7,56.6,72.9,79.3
576.0,0.3,0.2,0.67,55.7,42.2,62.2,65.6
577.0,0.73,0.25,0.33,48.1,53.7,38.2,59.3
578.0,-0.14,-0.17,0.67,45.6,34.8,38.0,39.0
579.0,-0.77,-1.24,0.0,33.8,54.2,38.4,36.1
580.0,-0.13,-0.19,0.0,38.3,43.2,59.2,62.6
581.0,0.34,-0.14,0.33,52.9,27.7,34.1,51.1
582.0,-0.37,-1.48,0.0,32.4,52.0,47.1,48.4
583.0,-0.8,0.69,0.0,45.2,41.0,54.4,50.6
584.0,-0.05,1.51,1.0,32.7,39.3,53.4,57.2
585.0,-0.61,-0.84,0.0,39.2,30.8,39.3,42.1
586.0,0.62,0.11,1.0,53.0,42.6,42.8,50.5
587.0,-0.55,0.71,0.67,69.7,57.8,57.9,54.2
588.0,-0.57,-0.23,0.67,53.8,47.6,61.2,48.4
589.0,-0.1,0.25,0.33,45.1,70.1,66.5,48.6
590.0,-0.09,0.5,0.0,47.7,45.4,53.1,55.7
591.0,0.81,0.57,1.0,49.9,47.5,64.2,55.2
592.0,0.35,-0.65,0.67,41.5,33.9,38.7,44.2
593.0,-0.68,-0.78,0.67,56.8,41.5,46.6,52.5
594.0,0.97,0.27,0.33,55.9,49.6,45.4,45.4
595.0,0.02,0.0,0.67,57.7,50.2,50.2,54.9
596.0,-0.15,-0.57,0.67,39.9,69.8,49.1,44.7
597.0,0.21,0.46,0.67,51.1,52.1,45.7,44.8
598.0,1.2,1.15,0.67,54.6,46.9,44.8,53.6
599.0,0.23,0.11,0.67,56.1,55.6,46.4,46.8
600.0,2.0,1.93,1.0,69.0,68.6,58.9,57.9
""")

## Helper: collinear_columns

A step that no library does, written out. In the code download it is the file `collinear_columns.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.multivariate.multivariate_ols import multivariate_stats
# collinear_columns: defined in the helper cell collinear_columns above (collinear_columns.py in the code download)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# Two sets of numeric variables; text in a numeric column counts as missing.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
first = ["Engagement", "Autonomy", "Training completed"]
second = ["Sales score", "Customer rating score", "Projects score", "Attendance score"]
names = [*first, *second]
frame = data[names].apply(pd.to_numeric, errors="coerce")

## Cases

In [ ]:
ok1, ok2 = frame[first].notna().all(axis=1), frame[second].notna().all(axis=1)
summary = pd.DataFrame(
    {"n": [(ok1 & ok2).sum(), (~ok1 & ok2).sum(), (ok1 & ~ok2).sum(), (~ok1 & ~ok2).sum(),
           len(frame)]},
    index=["valid", "first set missing", "second set missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)
valid = frame[ok1 & ok2]
n = len(valid)

descriptive = pd.DataFrame({"n": n, "mean": valid.mean(), "variance": valid.var(ddof=1),
                            "SD": valid.std(ddof=1), "min": valid.min(), "max": valid.max()})
correlation = valid.corr()
print()
print(descriptive)
print()
print(correlation)

## Variables that are linear combinations of others

In [ ]:
# Each set is checked on its own. A variable that is an exact linear combination of
# others in its own set makes that set's covariance matrix singular, and the weights need
# its inverse. A variable that the other set gives exactly is not a defect: it is a
# canonical correlation of 1.
dropped = [*collinear_columns(valid[first].to_numpy(float), first),
           *collinear_columns(valid[second].to_numpy(float), second)]
print("\nLinearly dependent:", dropped or "none")
# They are left out of the analysis.
first = [name for name in first if name not in dropped]
second = [name for name in second if name not in dropped]
x, y = valid[first].to_numpy(float), valid[second].to_numpy(float)

## Canonical correlations

In [ ]:
# Whiten each set (Sxx^-1/2, Syy^-1/2, from the n - 1 covariances); the singular values of
# Sxx^-1/2 Sxy Syy^-1/2 are the canonical correlations, its singular vectors give the
# weights. Each pair is signed so the second set's weights add up to a positive number.
xc, yc = x - x.mean(axis=0), y - y.mean(axis=0)
sxx, syy, sxy = xc.T @ xc / (n - 1), yc.T @ yc / (n - 1), xc.T @ yc / (n - 1)


def inverse_root(matrix):
    values, vectors = np.linalg.eigh(matrix)
    return vectors @ np.diag(values ** -0.5) @ vectors.T


root_x, root_y = inverse_root(sxx), inverse_root(syy)
u, r, vt = np.linalg.svd(root_x @ sxy @ root_y)
k = min(len(first), len(second))
r = np.clip(r[:k], 0, 1)
# A correlation with 1 - r^2 at most 1e-10 is 1: a combination of one set is an exact
# linear combination of the other (the tolerance collinear_columns() uses). Rounding
# leaves such a correlation at 0.9999999999999998 or so.
r[1 - r**2 <= 1e-10] = 1.0
a, b = root_x @ u[:, :k], root_y @ vt[:k].T
flip = np.where(b.sum(axis=0) < 0, -1.0, 1.0)
a, b = a * flip, b * flip

# At r = 1 the eigenvalue r^2 / (1 - r^2) is infinite, and no pair has a share of the sum.
with np.errstate(divide="ignore"):
    eigenvalues = r**2 / (1 - r**2)
share = np.full(k, np.nan) if np.isinf(eigenvalues).any() else eigenvalues / eigenvalues.sum()
eigen = pd.DataFrame({"variate": np.arange(1, k + 1), "eigenvalue": eigenvalues,
                      "share": share, "cumulative": np.cumsum(share), "r": r, "r2": r**2})
print()
print(eigen.to_string(index=False))

# Sequential tests (Rao's F on Wilks' lambda): is anything left from pair j onwards?
p, q = len(second), len(first)
df_error = n - q - 1
tests = []
for j in range(k):
    lam = np.prod(1 - r[j:] ** 2)
    pj, qj = p - j, q - j
    df1 = pj * qj
    numerator, denominator = pj**2 * qj**2 - 4, pj**2 + qj**2 - 5
    t = np.sqrt(numerator / denominator) if numerator > 0 and denominator > 0 else 1.0
    df2 = t * (df_error - (pj - qj + 1) / 2) - (df1 - 2) / 2
    f = (1 - lam ** (1 / t)) / lam ** (1 / t) * df2 / df1
    tests.append([lam, f, df1, df2, stats.f.sf(f, df1, df2)])
tests = pd.DataFrame(tests, columns=["lambda", "F", "df1", "df2", "P"])
print()
print(tests)

# The four multivariate tests of "no relationship at all", as for a multivariate
# regression of the second set on the first (Hotelling's trace in McKeon's F form).
multivariate = multivariate_stats(r**2, p, q, df_error).loc[
    ["Pillai's trace", "Wilks' lambda", "Hotelling-Lawley trace", "Roy's greatest root"],
    ["Value", "F Value", "Num DF", "Den DF", "Pr > F"]]
print()
print(multivariate)

## Weights

In [ ]:
# Raw weights make each canonical variate's variance 1; standardised weights apply to
# the variables in SD units.
numbers = [str(j + 1) for j in range(k)]
sd_x, sd_y = x.std(axis=0, ddof=1), y.std(axis=0, ddof=1)
coefficients_first = pd.DataFrame(a, index=first, columns=numbers)
coefficients_second = pd.DataFrame(b, index=second, columns=numbers)
standardised_first = coefficients_first.mul(sd_x, axis=0)
standardised_second = coefficients_second.mul(sd_y, axis=0)
print()
print(coefficients_first, coefficients_second, standardised_first, standardised_second, sep="\n\n")

scores_first = xc @ a
scores_second = yc @ b

## Canonical scores

In [ ]:
print()
print(pd.DataFrame(scores_first, columns=numbers).head(), pd.DataFrame(scores_second, columns=numbers).head(),
      sep="\n\n")

## Structure: loadings and cross-loadings

In [ ]:
# Loadings correlate each variable with its own set's variates; the mean square is the
# share of the set a variate carries. Cross-loadings correlate it with the other set's
# variates; their mean square is the redundancy, the share the other set predicts.


def structure(values, scores, names):
    loadings = np.array([[np.corrcoef(values[:, i], scores[:, j])[0, 1] for j in range(k)]
                         for i in range(values.shape[1])])
    share = (loadings**2).mean(axis=0)
    return pd.DataFrame(np.vstack([loadings, share, np.cumsum(share)]),
                        index=[*names, "share", "cumulative"], columns=numbers)


structure_first, structure_second = structure(x, scores_first, first), structure(y, scores_second, second)
cross_first, cross_second = structure(x, scores_second, first), structure(y, scores_first, second)
print()
print(structure_first, structure_second, cross_first, cross_second, sep="\n\n")

## What the result says

Canonical correlation analysis looks for the strongest link between two sets of measures, here three attitude measures and four performance scores for 600 employees. The first canonical correlation is 0.60, and the test of all three (Wilks' lambda 0.62, F = 25.8) gives a P value of almost zero: chance alone would almost never give a link this strong. The second correlation is only 0.19, and the third is not significant (P = 0.93).

The canonical structure shows what the first link is made of: engagement (0.86) and autonomy (0.77) on the attitude side, and sales (0.81) above all on the performance side. But the redundancy, the share of the variation in the performance scores that the attitudes account for, is only 0.148 in total.

So attitudes and performance are clearly linked, mainly through one channel, but the attitudes explain only a small part of performance.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.